# 3D vs 0D/1D Hemodynamic Model Comparison

**Goal:** Compare hydraulic resistance predicted by three model fidelity levels using the same geometry (patient MR 025, segment labels 1+2+8).

| Model | Description | Input from |
|---|---|---|
| **3D** | Full CFD (OpenFOAM simpleFoam) | `hemodynamics_analysis.ipynb` results |
| **0D** | Poiseuille resistance, mean radius | Centerline graph (`graph_mr_025.vtp`) |
| **1D** | Integrated variable-radius resistance | Centerline radius profile r(s) |

**Key equation — Poiseuille resistance:**
$$R_{0D} = \frac{8\,\mu\,L}{\pi\,\bar{r}^4}$$

**Key equation — 1D (variable cross-section):**
$$R_{1D} = \int_0^L \frac{8\,\mu}{\pi\,r(s)^4}\,ds$$

**Blood properties:** μ = 3.5×10⁻³ Pa·s, ρ = 1060 kg/m³

## Section 1 — Setup and Constants

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
from pathlib import Path

import vtk
from vtk.util.numpy_support import vtk_to_numpy

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT     = Path("..").resolve()
DATA_DIR = ROOT / "data"
CFD_DIR  = ROOT / "openfoam" / "segment_test_V2" / "data"

GRAPH_VTP   = DATA_DIR / "graph_mr_025.vtp"
SURFACE_VTP = DATA_DIR / "mr_limited" / "geometry" / "segment_test_surface.vtp"
FEATURES_JSON = DATA_DIR / "features_mr_025.json"

# ── Blood properties ───────────────────────────────────────────────────────
MU  = 3.5e-3    # dynamic viscosity [Pa·s]
RHO = 1060.0    # density [kg/m³]
NU  = MU / RHO  # kinematic viscosity [m²/s]  ≈ 3.30e-6

# ── CFD reference results (from hemodynamics_analysis.ipynb) ──────────────
# All values recomputed here directly from the CSV exports for reproducibility
iv = {k: pd.read_csv(CFD_DIR / f"{k}_integration_variable.csv")
      for k in ["inlet", "midslice", "outlet"]}

def extract_slice(df):
    row  = df.iloc[0]
    area = row["Area"]
    U    = np.array([row["U:0"], row["U:1"], row["U:2"]])
    Q    = np.linalg.norm(U)
    p_kn = row["p"] / area      # mean kinematic pressure [m²/s²]
    return dict(area=area, Q=Q, p_kin=p_kn, p_Pa=RHO * p_kn)

cfd = {k: extract_slice(iv[k]) for k in ["inlet", "midslice", "outlet"]}

Q_3D   = cfd["inlet"]["Q"]                           # [m³/s]
dP_3D  = cfd["inlet"]["p_Pa"] - cfd["outlet"]["p_Pa"]  # [Pa]
R_3D   = dP_3D / Q_3D                                 # [Pa·s/m³]

print(f"CFD reference (3D):  Q={Q_3D*1e6:.4f} mL/s  ΔP={dP_3D:.4f} Pa  R={R_3D:.4e} Pa·s/m³")
print(f"                     R={R_3D/133.322e6:.4f} mmHg·s/mL")

## Section 2 — Load Centerline Geometry

The graph VTP contains the Circle of Willis centerline skeleton with 12 labeled vessel segments.  
Each graph edge is a short line segment with an associated local vessel radius.

**Step:** identify which graph labels overlap the CFD bounding box → labels 1, 2, 8.  
Then extract only those edges and build the centerline used for 0D and 1D models.

In [ ]:
def load_graph(vtp_path: Path) -> dict:
    """Read a centerline graph VTP and return arrays of point coords, edge data."""
    reader = vtk.vtkXMLPolyDataReader()
    reader.SetFileName(str(vtp_path))
    reader.Update()
    pd = reader.GetOutput()

    pts    = vtk_to_numpy(pd.GetPoints().GetData())          # (N,3) mm
    labels = vtk_to_numpy(pd.GetCellData().GetArray("labels"))
    ce_r   = vtk_to_numpy(pd.GetCellData().GetArray("ce_radius"))   # mm
    mis_r  = vtk_to_numpy(pd.GetCellData().GetArray("mis_radius"))  # mm
    degree = vtk_to_numpy(pd.GetPointData().GetArray("degree"))

    # Edge endpoint indices
    n_cells = pd.GetNumberOfCells()
    edge_p0 = np.array([pd.GetCell(i).GetPointId(0) for i in range(n_cells)])
    edge_p1 = np.array([pd.GetCell(i).GetPointId(1) for i in range(n_cells)])

    return dict(pts=pts, labels=labels, ce_r=ce_r, mis_r=mis_r,
                degree=degree, edge_p0=edge_p0, edge_p1=edge_p1)


def graph_segment_stats(g: dict, label_mask: np.ndarray) -> dict:
    """Compute arc length and radius stats for a boolean edge mask."""
    p0 = g["pts"][g["edge_p0"][label_mask]]
    p1 = g["pts"][g["edge_p1"][label_mask]]
    edge_lengths = np.linalg.norm(p1 - p0, axis=1)   # mm
    radii = g["ce_r"][label_mask]                      # mm
    return dict(
        n_edges    = label_mask.sum(),
        arc_len_mm = edge_lengths.sum(),
        edge_len   = edge_lengths,
        radii_mm   = radii,
        r_mean_mm  = radii.mean(),
        r_std_mm   = radii.std(),
        r_min_mm   = radii.min(),
        r_max_mm   = radii.max(),
    )


# ── Load full graph ────────────────────────────────────────────────────────
g = load_graph(GRAPH_VTP)
print(f"Graph loaded: {len(g['pts'])} nodes, {len(g['labels'])} edges")
print(f"Labels present: {np.unique(g['labels']).tolist()}")
print()

# ── Summary table for all 12 labels ───────────────────────────────────────
rows = []
for lab in np.unique(g["labels"]):
    mask = g["labels"] == lab
    s = graph_segment_stats(g, mask)
    rows.append(dict(label=lab, n_edges=s["n_edges"],
                     arc_len_mm=round(s["arc_len_mm"], 3),
                     r_mean_mm=round(s["r_mean_mm"], 4),
                     r_std_mm=round(s["r_std_mm"], 4)))

df_labels = pd.DataFrame(rows)
print("All graph segments:")
print(df_labels.to_string(index=False))

In [ ]:
# ── Identify which labels overlap the CFD domain ──────────────────────────
surf_reader = vtk.vtkXMLPolyDataReader()
surf_reader.SetFileName(str(SURFACE_VTP))
surf_reader.Update()
surf_pts = vtk_to_numpy(surf_reader.GetOutput().GetPoints().GetData())

bb_margin = 1.0  # mm
xmin, xmax = surf_pts[:,0].min()-bb_margin, surf_pts[:,0].max()+bb_margin
ymin, ymax = surf_pts[:,1].min()-bb_margin, surf_pts[:,1].max()+bb_margin
zmin, zmax = surf_pts[:,2].min()-bb_margin, surf_pts[:,2].max()+bb_margin

print(f"CFD surface bounding box (+ {bb_margin} mm margin):")
print(f"  x=[{xmin:.2f}, {xmax:.2f}]  y=[{ymin:.2f}, {ymax:.2f}]  z=[{zmin:.2f}, {zmax:.2f}]  (mm)")

# Mark graph nodes inside the bbox
pts = g["pts"]
in_bbox = ((pts[:,0] >= xmin) & (pts[:,0] <= xmax) &
           (pts[:,1] >= ymin) & (pts[:,1] <= ymax) &
           (pts[:,2] >= zmin) & (pts[:,2] <= zmax))

# Edges where BOTH endpoints are inside the bbox
edge_in = in_bbox[g["edge_p0"]] & in_bbox[g["edge_p1"]]
cfd_labels = np.unique(g["labels"][edge_in])
print(f"\nGraph labels inside CFD domain: {cfd_labels.tolist()}")

# CFD-domain edge mask (within bbox, labels 1/2/8)
cfd_mask = edge_in & np.isin(g["labels"], cfd_labels)
cfd_seg  = graph_segment_stats(g, cfd_mask)

print(f"\nCFD-domain centerline summary:")
print(f"  Edges        : {cfd_seg['n_edges']}")
print(f"  Arc length   : {cfd_seg['arc_len_mm']:.3f} mm")
print(f"  r_ce  mean   : {cfd_seg['r_mean_mm']:.4f} mm")
print(f"  r_ce  std    : {cfd_seg['r_std_mm']:.4f} mm")
print(f"  r_ce  range  : [{cfd_seg['r_min_mm']:.4f}, {cfd_seg['r_max_mm']:.4f}] mm")

## Section 3 — 0D Poiseuille Model

The simplest reduced-order model treats the vessel segment as a straight rigid tube:

$$R_{0D} = \frac{8\,\mu\,L}{\pi\,\bar{r}^4}$$

where $\bar{r}$ is the harmonic mean of the radius (which gives a better estimate than the arithmetic mean for resistance in a variable-radius tube):

$$\bar{r}_{harm} = \left(\frac{1}{N}\sum_i \frac{1}{r_i^4}\right)^{-1/4}$$

We compute $R_{0D}$ three ways to show sensitivity to the radius estimate:
- **0D-arithmetic**: using arithmetic mean radius
- **0D-harmonic**: using harmonic mean (more accurate)
- **0D-features**: using the pre-computed mean radius from `features_mr_025.json`

In [ ]:
def poiseuille_R(L_m: float, r_m: float, mu: float = MU) -> float:
    """Poiseuille resistance R = 8μL / (π r⁴)  [Pa·s/m³]"""
    return 8 * mu * L_m / (np.pi * r_m**4)


L_m  = cfd_seg["arc_len_mm"] * 1e-3          # arc length [m]
r_mm = cfd_seg["radii_mm"]                    # per-edge radii [mm]
r_m  = r_mm * 1e-3                            # [m]

# ── Three 0D estimates ─────────────────────────────────────────────────────
r_arith = r_m.mean()
r_harm  = (np.mean(1.0 / r_m**4))**(-0.25)   # harmonic mean w.r.t. r⁴

R_0D_arith  = poiseuille_R(L_m, r_arith)
R_0D_harm   = poiseuille_R(L_m, r_harm)

# From features JSON (pre-computed, variant 1 — check which segments match)
with open(FEATURES_JSON) as f:
    features = json.load(f)

# Collect segment entries whose graph label appears in cfd_labels
feat_rows = []
v1 = features.get("1", {})
for seg_name, seg_list in v1.items():
    if not isinstance(seg_list, list):
        continue
    for entry in seg_list:
        if "segment" not in entry:
            continue
        feat_rows.append(dict(
            name    = seg_name,
            L_mm    = entry["length"],
            r_mean  = entry["radius"]["mean"],
        ))

df_feat = pd.DataFrame(feat_rows)
print("Features JSON segments (variant 1):")
print(df_feat.to_string(index=False))
print()

# ── Display 0D results ─────────────────────────────────────────────────────
conv = 1 / 133.322e6   # Pa·s/m³ → mmHg·s/mL

print(f"Arc length L     = {L_m*1e3:.3f} mm")
print(f"r arithmetic     = {r_arith*1e3:.4f} mm")
print(f"r harmonic-mean  = {r_harm*1e3:.4f} mm")
print()
print(f"{'Model':<22} {'R [Pa·s/m³]':>16} {'R [mmHg·s/mL]':>16}")
print("-" * 56)
print(f"{'0D (arith. mean r)':<22} {R_0D_arith:>16.4e} {R_0D_arith*conv:>16.4f}")
print(f"{'0D (harmonic r)':<22} {R_0D_harm:>16.4e} {R_0D_harm*conv:>16.4f}")
print(f"{'3D (CFD reference)':<22} {R_3D:>16.4e} {R_3D*conv:>16.4f}")

## Section 4 — 1D Model (Variable Radius Integration)

The 1D model integrates the Poiseuille resistance element-by-element along the centerline.  
Each edge $i$ with length $ds_i$ and radius $r_i$ contributes:

$$dR_i = \frac{8\,\mu\,ds_i}{\pi\,r_i^4}$$

Total: $R_{1D} = \sum_i dR_i$

This is equivalent to assuming steady laminar flow in each infinitesimal cross-section — it captures the effect of vessel narrowing (stenoses) without solving the 3D flow equations.  
The 1D result approaches the 0D result only when $r(s) = \text{const}$.

We also compute the cumulative resistance profile $R(s)$ to see where most of the resistance is concentrated.

In [ ]:
# ── Build ordered centerline path through the CFD domain ──────────────────
# Extract the sub-graph edges inside the CFD domain
e0_cfd = g["edge_p0"][cfd_mask]
e1_cfd = g["edge_p1"][cfd_mask]
r_cfd  = g["ce_r"][cfd_mask] * 1e-3         # [m]
pts_cfd = g["pts"]                           # all points [mm]

p0_cfd = pts_cfd[e0_cfd] * 1e-3             # [m]
p1_cfd = pts_cfd[e1_cfd] * 1e-3             # [m]
ds_cfd = np.linalg.norm(p1_cfd - p0_cfd, axis=1)  # edge lengths [m]

# Per-edge resistance contribution
dR = 8 * MU * ds_cfd / (np.pi * r_cfd**4)

R_1D = dR.sum()

# Cumulative arc length and cumulative resistance for profile plot
# Sort edges by position along the dominant axis (y-axis based on extent)
centroid_y = ((p0_cfd + p1_cfd) / 2)[:, 1]  # mid-point y coord [m]
sort_idx   = np.argsort(centroid_y)

s_sorted   = np.cumsum(ds_cfd[sort_idx]) * 1e3        # [mm]
r_sorted   = r_cfd[sort_idx] * 1e3                     # [mm]
dR_sorted  = dR[sort_idx]
cumR_sorted = np.cumsum(dR_sorted)

print(f"1D model: R_1D = {R_1D:.4e} Pa·s/m³  ({R_1D*conv:.4f} mmHg·s/mL)")
print(f"0D arith: R_0D = {R_0D_arith:.4e} Pa·s/m³  ({R_0D_arith*conv:.4f} mmHg·s/mL)")
print(f"3D CFD:   R_3D = {R_3D:.4e} Pa·s/m³  ({R_3D*conv:.4f} mmHg·s/mL)")
print()
print(f"Number of 1D elements: {len(dR)}")
print(f"ds range: [{ds_cfd.min()*1e3:.4f}, {ds_cfd.max()*1e3:.4f}] mm")
print(f"r  range: [{r_cfd.min()*1e3:.4f}, {r_cfd.max()*1e3:.4f}] mm")

## Section 5 — Comparison and Error Metrics

$$\epsilon_{rel}(M) = \frac{R_M - R_{3D}}{R_{3D}} \times 100\%$$

A positive $\epsilon$ means the reduced model **over-estimates** resistance (flow is harder to drive than CFD predicts).  
A negative $\epsilon$ means the reduced model **under-estimates** resistance.

In [ ]:
models = {
    "3D CFD":         R_3D,
    "0D (arith. r)":  R_0D_arith,
    "0D (harmonic r)":R_0D_harm,
    "1D (variable r)":R_1D,
}

print(f"{'Model':<22} {'R [Pa·s/m³]':>16} {'R [mmHg·s/mL]':>16} {'ε_rel [%]':>12}")
print("-" * 70)
for name, R in models.items():
    err = (R - R_3D) / R_3D * 100
    flag = "" if name == "3D CFD" else f"  {'+' if err>=0 else ''}{err:.1f}%"
    print(f"{name:<22} {R:>16.4e} {R*conv:>16.4f} {flag:>12}")

# Summary DataFrame for further use / export
df_results = pd.DataFrame([
    {"Model": name, "R_Pa_s_m3": R, "R_mmHg_s_mL": R*conv,
     "epsilon_pct": (R-R_3D)/R_3D*100}
    for name, R in models.items()
])
print()
print("Saved as df_results")

## Section 6 — Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("3D vs 0D/1D Resistance Comparison — MR 025 (labels 1+2+8)", fontsize=13, fontweight="bold")

# ── Plot A: Bar chart of R values ──────────────────────────────────────────
ax = axes[0]
model_names  = list(models.keys())
r_values_mmhg = [R * conv for R in models.values()]
bar_colors   = ["#2196F3", "#FF9800", "#FF9800", "#4CAF50"]
bar_alpha    = [1.0, 0.7, 0.85, 0.9]

bars = ax.bar(model_names, r_values_mmhg,
              color=bar_colors, alpha=0.85, edgecolor="black", linewidth=0.7)
ax.axhline(R_3D * conv, color="#2196F3", linestyle="--", linewidth=1.2, alpha=0.6)
for bar, val in zip(bars, r_values_mmhg):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(r_values_mmhg)*0.015,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_ylabel("R [mmHg·s/mL]")
ax.set_title("Hydraulic Resistance")
ax.tick_params(axis="x", rotation=20)
ax.grid(True, axis="y", alpha=0.3)

# ── Plot B: Relative error bar ─────────────────────────────────────────────
ax = axes[1]
errs = [(R - R_3D)/R_3D*100 for R in models.values()]
err_colors = ["#2196F3" if abs(e) < 1e-9 else ("#E53935" if e > 0 else "#43A047")
              for e in errs]
ax.bar(model_names, errs, color=err_colors, alpha=0.85, edgecolor="black", linewidth=0.7)
ax.axhline(0, color="black", linewidth=1.0)
for i, (name, err) in enumerate(zip(model_names, errs)):
    if abs(err) > 0.01:
        ax.text(i, err + np.sign(err)*2, f"{err:+.1f}%", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Relative error vs 3D [%]")
ax.set_title("Error: (R_model − R_3D) / R_3D")
ax.tick_params(axis="x", rotation=20)
ax.grid(True, axis="y", alpha=0.3)

# ── Plot C: 1D radius profile and cumulative resistance along centerline ───
ax = axes[2]
ax2 = ax.twinx()

ax.plot(s_sorted, r_sorted, color="#4CAF50", linewidth=1.5, label="r(s) [mm]")
ax.fill_between(s_sorted, r_sorted, alpha=0.2, color="#4CAF50")
ax2.plot(s_sorted, cumR_sorted * conv, color="#E53935", linewidth=1.5, linestyle="-.",
         label="Cumulative R [mmHg·s/mL]")

ax.set_xlabel("Arc length s [mm]")
ax.set_ylabel("Vessel radius r(s) [mm]", color="#4CAF50")
ax2.set_ylabel("Cumulative R [mmHg·s/mL]", color="#E53935")
ax.set_title("1D Radius Profile & Cumulative Resistance")
ax.tick_params(axis="y", labelcolor="#4CAF50")
ax2.tick_params(axis="y", labelcolor="#E53935")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="upper left")
ax.grid(True, alpha=0.3)

plt.tight_layout()
out_path = CFD_DIR / "model_comparison_summary.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

## Section 7 — Full Graph: 0D Resistance for All 12 Segments

Compute $R_{0D}$ for every labeled segment across the whole CoW graph.  
This gives a complete picture of the vascular network resistance and provides  
inputs for a full 0D network model (each segment becomes a resistor in a circuit).

In [ ]:
network_rows = []
for lab in np.unique(g["labels"]):
    mask = g["labels"] == lab
    s    = graph_segment_stats(g, mask)

    L_m_seg = s["arc_len_mm"] * 1e-3
    r_m_seg = s["radii_mm"] * 1e-3

    R0_arith = poiseuille_R(L_m_seg, r_m_seg.mean())
    R0_harm  = poiseuille_R(L_m_seg, (np.mean(1/r_m_seg**4))**(-0.25))
    R1d      = np.sum(8 * MU * (np.linalg.norm(
                   g["pts"][g["edge_p0"][mask]] - g["pts"][g["edge_p1"][mask]], axis=1
               ) * 1e-3) / (np.pi * r_m_seg**4))

    is_cfd = lab in cfd_labels
    network_rows.append(dict(
        label       = lab,
        arc_len_mm  = round(s["arc_len_mm"], 2),
        r_mean_mm   = round(s["r_mean_mm"], 4),
        R0_arith    = R0_arith,
        R0_harm     = R0_harm,
        R_1D        = R1d,
        in_CFD_domain = "★" if is_cfd else "",
    ))

df_network = pd.DataFrame(network_rows)
df_network["R0_arith_mmHg"] = df_network["R0_arith"] * conv
df_network["R0_harm_mmHg"]  = df_network["R0_harm"]  * conv
df_network["R_1D_mmHg"]     = df_network["R_1D"]     * conv

cols_display = ["label","arc_len_mm","r_mean_mm","R0_arith_mmHg","R0_harm_mmHg","R_1D_mmHg","in_CFD_domain"]
print("Full network 0D/1D resistance table [mmHg·s/mL]:")
print(df_network[cols_display].to_string(index=False))

# Save to CSV
out_csv = CFD_DIR / "network_resistance_table.csv"
df_network.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

## Section 8 — Straight Pipe Validation (Poiseuille Benchmark)

Before trusting the hemodynamics 3D results, we validate the **solver stack** against the exact analytical solution for laminar pipe flow (Hagen-Poiseuille).

**Case:** `openfoam/pipe/`  
- Geometry: R = 2 mm, L = 40 mm (L/D = 10), pipe axis = z  
- Blood: μ = 3.5×10⁻³ Pa·s, ρ = 1060 kg/m³, ν = 3.3×10⁻⁶ m²/s  
- Flow: U_mean = 0.1 m/s → Re = 121 (fully laminar)  
- Inlet BC: parabolic (fully-developed Poiseuille profile via `codedFixedValue`)  
- Solver: same `simpleFoam` + `fvSchemes`/`fvSolution` as `segment_test_V2`

**Analytical reference:**

$$\Delta P = \frac{8\,\mu\,L\,\bar{U}}{R^2} = 28\text{ Pa}, \qquad
u_z(r) = 2\bar{U}\!\left(1 - \frac{r^2}{R^2}\right), \qquad U_{max} = 2\bar{U} = 0.2\text{ m/s}$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
PIPE_DIR  = ROOT / "openfoam" / "pipe"
PIPE_POST = sorted((PIPE_DIR / "postProcessing" / "sampleDict").iterdir())[-1]

# ── Pipe geometry and blood properties (same as main notebook) ─────────────
R_pipe   = 0.002    # m  (radius)
L_pipe   = 0.040    # m  (length)
U_mean   = 0.1      # m/s
RHO_pipe = 1060.0   # kg/m³
MU_pipe  = 3.5e-3   # Pa·s
NU_pipe  = MU_pipe / RHO_pipe

# ── Analytical Poiseuille reference ───────────────────────────────────────
dP_analytical = 8 * MU_pipe * L_pipe * U_mean / R_pipe**2   # Pa
R_analytical  = 8 * MU_pipe * L_pipe / (np.pi * R_pipe**4)  # Pa·s/m³
U_max_an      = 2 * U_mean                                    # m/s

# ── Load CFD patch-average (run postProcess to regenerate if needed) ───────
# p_kin_inlet from: postProcess -func "patchAverage(patch=inlet, fields=(p))"
p_kin_inlet_CFD = 0.026884357   # m²/s²  (from latest run)
dP_CFD = p_kin_inlet_CFD * RHO_pipe   # Pa

Q_pipe = np.pi * R_pipe**2 * U_mean   # m³/s
R_CFD  = dP_CFD / Q_pipe              # Pa·s/m³

conv = 1 / 133.322e6   # Pa·s/m³ → mmHg·s/mL

# ── Load sampled radial profile near outlet ────────────────────────────────
# Columns: distance  x  y  z  Ux  Uy  Uz  p
radial_file = PIPE_POST / "outletRadial.xy"
rad_data = np.loadtxt(radial_file, comments="#")
x_rad  = rad_data[:, 1]   # radial coordinate [m]
Uz_CFD = rad_data[:, 6]   # axial velocity    [m/s]

# Analytical profile at the same x positions
r2 = x_rad**2
Uz_an = U_max_an * np.maximum(0.0, 1.0 - r2 / R_pipe**2)

# ── Load sampled centreline (axial pressure profile) ──────────────────────
centreline_file = PIPE_POST / "centreline.xy"
cl_data = np.loadtxt(centreline_file, comments="#")
z_cl  = cl_data[:, 3]   # z coordinate [m]
p_cl  = cl_data[:, 7]   # kinematic pressure [m²/s²]

# Analytical pressure along centreline: p(z) = dP_an/L * (L - z) / rho
p_cl_an = (dP_analytical / L_pipe) * (L_pipe - z_cl) / RHO_pipe

# ── Validation error metrics ───────────────────────────────────────────────
err_dP   = (dP_CFD - dP_analytical) / dP_analytical * 100
err_Umax = (Uz_CFD.max() - U_max_an) / U_max_an * 100

# Profile pointwise error (ignore near-wall where Uz < 1% of Umax)
mask_core = Uz_an > 0.01 * U_max_an
profile_err_mean = np.abs((Uz_CFD[mask_core] - Uz_an[mask_core]) / Uz_an[mask_core] * 100).mean()

print("=" * 58)
print(f"  Poiseuille pipe validation — R={R_pipe*1e3:.0f}mm, L={L_pipe*1e3:.0f}mm, Re={U_mean*2*R_pipe/NU_pipe:.0f}")
print("=" * 58)
print(f"  {'Quantity':<32} {'CFD':>10} {'Analytical':>10} {'err%':>7}")
print(f"  {'-'*59}")
print(f"  {'ΔP [Pa]':<32} {dP_CFD:>10.4f} {dP_analytical:>10.4f} {err_dP:>6.2f}%")
print(f"  {'R [mmHg·s/mL]':<32} {R_CFD*conv:>10.5f} {R_analytical*conv:>10.5f} {err_dP:>6.2f}%")
print(f"  {'U_max [m/s]':<32} {Uz_CFD.max():>10.5f} {U_max_an:>10.5f} {err_Umax:>6.2f}%")
print(f"  {'Profile shape (mean |err|)':<32} {'':>10} {'':>10} {profile_err_mean:>6.2f}%")
print()
print(f"  Solver: simpleFoam + laminar + same fvSchemes/fvSolution as segment_test_V2")
print(f"  ✓  Solver validated: ΔP error {err_dP:.1f}%,  U_max error {err_Umax:.2f}%")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Straight Pipe Validation — Poiseuille Benchmark (R=2 mm, Re=121)",
             fontsize=13, fontweight="bold")

# ── Plot A: Radial velocity profile ──────────────────────────────────────
ax = axes[0]
ax.plot(x_rad * 1e3, Uz_CFD, "b-",  linewidth=2,   label="CFD (simpleFoam)")
ax.plot(x_rad * 1e3, Uz_an,  "r--", linewidth=1.5,  label="Analytical Poiseuille")
ax.set_xlabel("Radial position x [mm]")
ax.set_ylabel("Axial velocity $U_z$ [m/s]")
ax.set_title("Radial Velocity Profile (near outlet)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.axvline(0, color="gray", linewidth=0.5, linestyle=":")
ax.text(0, U_max_an * 1.02, f"$U_{{max}}$ = {Uz_CFD.max():.4f} m/s\n(err = {err_Umax:.2f}%)",
        ha="center", va="bottom", fontsize=8, color="blue")

# ── Plot B: Axial pressure profile along centreline ───────────────────────
ax = axes[1]
ax.plot(z_cl * 1e3, p_cl  * RHO_pipe, "b-",  linewidth=2,   label="CFD (simpleFoam)")
ax.plot(z_cl * 1e3, p_cl_an * RHO_pipe, "r--", linewidth=1.5, label="Analytical")
ax.set_xlabel("Axial position z [mm]")
ax.set_ylabel("Pressure P [Pa]")
ax.set_title("Centreline Pressure Profile")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── Plot C: Resistance comparison — pipe (validated) vs vessel models ─────
ax = axes[2]
labels_bar  = ["Pipe CFD\n(validated)", "Pipe\nanalytical", "Vessel\n3D CFD",
               "Vessel\n0D (harm.)", "Vessel\n1D"]
R_bar       = [R_CFD, R_analytical, R_3D, R_0D_harm, R_1D]
R_bar_mmhg  = [r * conv for r in R_bar]
colors_bar  = ["#2196F3", "#90CAF9", "#FF5722", "#FF9800", "#4CAF50"]
hatches     = ["", "//", "", "", ""]

bars = ax.bar(labels_bar, R_bar_mmhg, color=colors_bar,
              edgecolor="black", linewidth=0.7, alpha=0.85)
for bar, h in zip(bars, hatches):
    bar.set_hatch(h)
for bar, val in zip(bars, R_bar_mmhg):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + max(R_bar_mmhg) * 0.01,
            f"{val:.4f}", ha="center", va="bottom", fontsize=8, fontweight="bold")
ax.set_ylabel("R [mmHg·s/mL]")
ax.set_title("Resistance — Pipe (benchmark) vs Vessel Models")
ax.tick_params(axis="x", rotation=15)
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
out_path = PIPE_DIR / "pipe_validation_summary.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")
